In [ ]:
import sys
sys.path.append('../')
from core import LSTransferTreeBoost, MTransferTreeBoost, LADTransferTreeBoost
import pandas as pd
import numpy as np

from friedman1 import *

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
#To run gammahat analysis and obtain magnitudes
df_hat = pd.DataFrame(columns = ['iteration', 'average_magnitude', 'd'])
df = pd.DataFrame(columns = ['iteration', 'average_magnitude', 'd'])
for d in [5, 20, 60]:
    for i in range(100):

        X_target_train, y_target_train = friedman1(n_samples=300, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=i) #no added noise
        X_source_train, y_source_train = friedman1_altered(n_samples=1000, add_noise = False, noise_distribution = 'gaussian', #no added noise
                                                            n_features=10, d=d, shift_seed=i, random_seed = i+1)

        #also add noise to source (only train here)

        average_magnitude_hat = []
        average_magnitude = []

        fiter = LSTransferTreeBoost(epochs=300, v=0.1, source_tree_size=2, #Or LAD or M
                                target_tree_size=2, k=0.0, m_0=0.5)
        leaf_gammas_tray, leaf_gammashats_tray, model_tray_clf, model_tray_clfhat, alpha_tray = fiter.fit(X_target_train, 
                                                                            y_target_train, X_source_train, y_source_train)
        


        iteration = 1
        for gammahats in leaf_gammashats_tray:
            average_magnitude_hat = np.mean([np.abs(elem) for elem in gammahats])
            df_hat.loc[len(df_hat)] = [iteration, average_magnitude_hat, d]
            iteration += 1
        iteration = 1
        for gammas in leaf_gammas_tray:
            average_magnitude = np.mean([np.abs(elem) for elem in gammas])
            df.loc[len(df)] = [iteration, average_magnitude, d]
            iteration += 1

        print(f'iteration {i} complete')

df_hat['q'] = df_hat['average_magnitude'] / df['average_magnitude'] #define q
df_hat.to_csv('vizes/df_hat_ls.csv') #save here

In [ ]:
#Visualize and save figure here!!
plt.figure(figsize = (17,4))
ls = pd.read_csv('vizes/df_hat_ls.csv')
ls['d'] = ls['d'].astype(int)
lad = pd.read_csv('vizes/df_hat_lad.csv')
lad['d'] = lad['d'].astype(int)
m = pd.read_csv('vizes/df_hat_m.csv')
m['d'] = m['d'].astype(int)

plt.subplot(1,3,1)
sns.lineplot(data = ls, x = 'iteration', y = 'q', hue = 'd', palette=["black", "purple", "red"])
plt.title('LS', fontsize = 14)
plt.ylabel('$q_{m}$', fontsize=12)
plt.xlabel('$m$', fontsize=12)
plt.ylim([0,0.5])
plt.legend(title=r'$d$')
plt.subplot(1,3,2)
sns.lineplot(data = lad, x = 'iteration', y = 'q', hue = 'd', palette=["black", "purple", "red"])
plt.title('LAD', fontsize = 14)
plt.ylabel('$q_{m}$', fontsize=12)
plt.xlabel('$m$', fontsize=12)
plt.ylim([0,0.5])
plt.legend(title=r'$d$')
plt.subplot(1,3,3)
sns.lineplot(data = m, x = 'iteration', y = 'q', hue = 'd', palette=["black", "purple", "red"])
plt.title('M', fontsize = 14)
plt.ylabel('$q_{m}$', fontsize=12)
plt.xlabel('$m$', fontsize=12)
plt.ylim([0,0.5])
plt.legend(title=r'$d$')
plt.savefig('vizes/magnitudes.png', bbox_inches = 'tight', pad_inches = 0.1, dpi = 300)
